# TTS WER Evaluation — VieNeu-TTS (Streaming Simulation)

Pipeline tái tạo đúng production:
```
question → Qwen3-4B (batch) → llm_answer → Brain chunker (word-safe, ≥40 chars)
        → TTS.infer(chunk_1) → audio_1
        → TTS.infer(chunk_2) → audio_2  
        → ...concat... → Whisper large-v3 → WER
```

**Models:**
- LLM: `Qwen/Qwen3-4B-2507-Instruct` (vLLM offline, thinking OFF)
- TTS: `pnnbao-ump/VieNeu-TTS-0.3B-q4-gguf` + `neuphonic/distill-neucodec`
- ASR Oracle: `openai/whisper-large-v3`

## 1. Cài đặt dependencies

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'Không có GPU — chạy CPU (chậm hơn)')

In [ ]:
!pip install -q vieneu neucodec
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install -q openai-whisper
!pip install -q jiwer soundfile librosa
!pip install -q vllm
print('Done!')

## 2. Cấu hình

In [ ]:
import os, torch

# ── LLM ─────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-2507-Instruct"

# ── Eval data ────────────────────────────────────
EVAL_FILE    = "eval_all.jsonl"
OUTPUT_DIR   = "tts_wer_output"
RESULTS_FILE = "tts_wer_results.jsonl"

MAX_SAMPLES  = 100   # None = toàn bộ 604 câu
RANDOM_SEED  = 42

# ── TTS ──────────────────────────────────────────
TTS_BACKBONE = "pnnbao-ump/VieNeu-TTS-0.3B-q4-gguf"
TTS_CODEC    = "neuphonic/distill-neucodec"

# ── Brain chunker (phải khớp production) ─────────
CHUNK_MIN_SIZE  = 40    # brain/core/chunker.py: MIN_CHUNK_SIZE
CHUNK_SILENCE_S = 0.15  # giây silence giữa các chunk audio

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device: {DEVICE}')
print(f'LLM:    {MODEL_ID}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Brain chunker (copy từ production)

In [ ]:
import re

PUNCTUATION = re.compile(r"[.!?;:,।।]")

def simulate_llm_stream(text: str, token_words: int = LLM_TOKEN_WORDS):
    """Giả lập LLM nhả token nhỏ từng đợt (word-level)."""
    words = text.split()
    for i in range(0, len(words), token_words):
        yield ' '.join(words[i:i + token_words]) + ' '


def apply_brain_chunker(text: str, min_size: int = CHUNK_MIN_SIZE) -> list[str]:
    """
    Tái tạo logic brain/core/chunker.py (sync version).
    Cắt tại dấu câu khi buffer >= min_size, fallback cắt tại space.
    """
    chunks = []
    buffer = ""

    for token in simulate_llm_stream(text):
        token = re.sub(r"\n+", " ", token)
        buffer += token

        while len(buffer) >= min_size:
            match = None
            for m in PUNCTUATION.finditer(buffer):
                if m.start() >= min_size // 2:
                    match = m

            if match:
                cut_pos = match.end()
                chunks.append(buffer[:cut_pos].strip())
                buffer = buffer[cut_pos:].lstrip()
            elif len(buffer) > min_size * 2:
                space_pos = buffer.rfind(" ", min_size // 2, min_size * 2)
                if space_pos > 0:
                    chunks.append(buffer[:space_pos].strip())
                    buffer = buffer[space_pos:].lstrip()
                else:
                    break
            else:
                break

    if buffer.strip():
        chunks.append(buffer.strip())

    return chunks


# Test chunker
test_text = "Vitamin C giúp tăng cường hệ miễn dịch. Nó có nhiều trong cam, chanh và ổi. Người lớn cần khoảng 75-90mg mỗi ngày."
test_chunks = apply_brain_chunker(test_text)
print(f'Input: {test_text}')
print(f'Chunks ({len(test_chunks)}):')
for i, c in enumerate(test_chunks):
    print(f'  [{i+1}] ({len(c)} chars) "{c}"')

## 4. Load data

In [ ]:
import json, random

records = []
with open(EVAL_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Tổng: {len(records)} câu')

if MAX_SAMPLES and MAX_SAMPLES < len(records):
    random.seed(RANDOM_SEED)
    records = random.sample(records, MAX_SAMPLES)
    print(f'Sample: {len(records)} câu')

## 5. Load Qwen3 + Sinh LLM answer

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

print(f'Loading {MODEL_ID} với vLLM...')
llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    max_model_len=8192,
    gpu_memory_utilization=0.85,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

sampling_params = SamplingParams(
    temperature=0.3,
    max_tokens=400,
)

SYSTEM_PROMPT = """Bạn là chuyên gia tư vấn dinh dưỡng. Trả lời câu hỏi bằng tiếng Việt.
Yêu cầu: tối đa 150 từ, KHÔNG dùng markdown, KHÔNG dùng số thứ tự hay gạch đầu dòng,
viết thành đoạn văn liền mạch tự nhiên như đang nói chuyện."""

print('Model ready!')

In [ ]:
import json
from tqdm.notebook import tqdm

llm_answers_file = os.path.join(OUTPUT_DIR, "llm_answers.jsonl")

# Resume: load những answers đã generate
done = {}
if os.path.exists(llm_answers_file):
    with open(llm_answers_file) as f:
        for line in f:
            item = json.loads(line)
            done[item['id']] = item
    print(f'Resume: {len(done)} answers đã có...')

todo = [r for r in records if r['id'] not in done]
print(f'Cần generate: {len(todo)} câu')

if todo:
    # Build prompts với chat template Qwen3 (tắt thinking mode)
    prompts = []
    for rec in todo:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": rec["question"]},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        prompts.append(text)

    # Batch inference
    outputs = llm.generate(prompts, sampling_params)

    with open(llm_answers_file, 'a') as f:
        for rec, output in zip(todo, outputs):
            answer = output.outputs[0].text.strip()
            entry = {'id': rec['id'], 'question': rec['question'], 'llm_answer': answer}
            done[rec['id']] = entry
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

qa_pairs = list(done.values())
print(f'Tổng answers: {len(qa_pairs)}')
avg_words = sum(len(x['llm_answer'].split()) for x in qa_pairs) // max(len(qa_pairs), 1)
print(f'Avg length: {avg_words} từ')

# Preview
sample = qa_pairs[0]
sample_chunks = apply_brain_chunker(sample['llm_answer'])
print(f'\nVí dụ: "{sample["question"][:60]}"')
print(f'Answer: {sample["llm_answer"][:120]}...')
print(f'Chunks ({len(sample_chunks)}): {[c[:40] for c in sample_chunks]}')

## 6. Khởi tạo VieNeu-TTS

In [ ]:
from vieneu import VieNeuTTS

print(f'Loading VieNeu-TTS trên {DEVICE}...')
tts = VieNeuTTS(
    backbone_repo=TTS_BACKBONE,
    backbone_device='gpu' if DEVICE == 'cuda' else 'cpu',
    codec_repo=TTS_CODEC,
    codec_device=DEVICE,
)
SAMPLE_RATE = tts.sample_rate  # 24000
print(f'TTS ready! Sample rate: {SAMPLE_RATE}Hz')
print(f'Voices: {tts.list_preset_voices()}')

## 7. Tổng hợp audio — streaming simulation

In [ ]:
import numpy as np
import soundfile as sf

SILENCE_SAMPLES = int(CHUNK_SILENCE_S * SAMPLE_RATE)
SILENCE_PAD     = np.zeros(SILENCE_SAMPLES, dtype=np.float32)


def synthesize_streaming(text: str, output_path: str) -> dict:
    """
    Tái tạo đúng production pipeline:
    1. Apply Brain chunker trên full text (simulate LLM stream)
    2. TTS.infer() từng chunk riêng
    3. Nối audio với silence 150ms giữa các chunk
    """
    chunks = apply_brain_chunker(text)
    if not chunks:
        return {'ok': False, 'n_chunks': 0}

    audio_parts = []
    failed_chunks = []

    for i, chunk in enumerate(chunks):
        try:
            # Mỗi chunk được synthesize độc lập — giống production
            audio = tts.infer(text=chunk)
            audio_parts.append(audio)
            if i < len(chunks) - 1:
                audio_parts.append(SILENCE_PAD)
        except Exception as e:
            failed_chunks.append({'chunk': i, 'text': chunk[:50], 'error': str(e)})

    if not audio_parts:
        return {'ok': False, 'n_chunks': len(chunks), 'failed': failed_chunks}

    final_audio = np.concatenate(audio_parts)
    sf.write(output_path, final_audio, SAMPLE_RATE)

    return {
        'ok': True,
        'n_chunks': len(chunks),
        'chunks': chunks,
        'duration_s': round(len(final_audio) / SAMPLE_RATE, 2),
        'failed_chunks': failed_chunks,
    }


# Test
test_path = os.path.join(OUTPUT_DIR, '_test.wav')
info = synthesize_streaming(qa_pairs[0]['llm_answer'], test_path)
print(f'Test: {info["n_chunks"]} chunks, {info["duration_s"]}s')
print('Chunks:')
for i, c in enumerate(info['chunks']):
    print(f'  [{i+1}] "{c[:70]}"')

from IPython.display import Audio
Audio(test_path)

In [ ]:
audio_dir = os.path.join(OUTPUT_DIR, 'audio')
os.makedirs(audio_dir, exist_ok=True)

tts_results = []
failed_tts  = []

for qa in tqdm(qa_pairs, desc='Synthesizing (streaming)'):
    audio_path = os.path.join(audio_dir, f"{qa['id']}.wav")
    meta_path  = audio_path.replace('.wav', '_meta.json')

    # Resume nếu đã có
    if os.path.exists(audio_path) and os.path.exists(meta_path):
        with open(meta_path) as mf:
            meta = json.load(mf)
        tts_results.append({**qa, 'audio_path': audio_path, **meta})
        continue

    info = synthesize_streaming(qa['llm_answer'], audio_path)
    if info['ok']:
        with open(meta_path, 'w') as mf:
            json.dump(info, mf, ensure_ascii=False)
        tts_results.append({**qa, 'audio_path': audio_path, **info})
    else:
        failed_tts.append(qa['id'])

print(f'\nTTS OK: {len(tts_results)} / {len(qa_pairs)}')
if failed_tts:
    print(f'Failed: {failed_tts}')

avg_chunks = sum(r['n_chunks'] for r in tts_results) / len(tts_results)
print(f'Avg chunks/answer: {avg_chunks:.1f}')

## 8. Transcribe bằng Whisper large-v3

In [ ]:
import whisper

print('Loading Whisper large-v3...')
whisper_model = whisper.load_model('large-v3', device=DEVICE)
print('Whisper ready!')

In [ ]:
def transcribe(audio_path: str) -> str:
    try:
        result = whisper_model.transcribe(
            audio_path,
            language='vi',
            task='transcribe',
            fp16=(DEVICE == 'cuda'),
        )
        return result['text'].strip()
    except Exception as e:
        print(f'  Whisper error: {e}')
        return ''

for item in tqdm(tts_results, desc='Transcribing'):
    if 'transcript' not in item:
        item['transcript'] = transcribe(item['audio_path'])

print('Transcription done!')

## 9. Tính WER

In [ ]:
from jiwer import wer, cer

def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

results = []
for item in tts_results:
    ref = normalize(item['llm_answer'])
    hyp = normalize(item.get('transcript', ''))
    if not ref or not hyp:
        continue

    results.append({
        'id':         item['id'],
        'question':   item['question'],
        'llm_answer': item['llm_answer'],
        'transcript': item.get('transcript', ''),
        'n_chunks':   item.get('n_chunks', 0),
        'duration_s': item.get('duration_s', 0),
        'wer':        round(wer(ref, hyp), 4),
        'cer':        round(cer(ref, hyp), 4),
        'ref_words':  len(ref.split()),
    })

with open(RESULTS_FILE, 'w') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f'Lưu {len(results)} kết quả vào {RESULTS_FILE}')

## 10. Tổng hợp kết quả

In [ ]:
wers = [r['wer'] for r in results]
cers = [r['cer'] for r in results]

print('=' * 55)
print('KẾT QUẢ TTS WER — VieNeu-TTS-0.3B-q4-gguf (Streaming)')
print('=' * 55)
print(f'Số mẫu:           {len(results)}')
print(f'Avg chunks/answer:{sum(r["n_chunks"] for r in results)/len(results):.1f}')
print(f'')
print(f'WER trung bình:   {np.mean(wers):.4f}  ({np.mean(wers)*100:.2f}%)')
print(f'WER trung vị:     {np.median(wers):.4f}  ({np.median(wers)*100:.2f}%)')
print(f'WER std:          {np.std(wers):.4f}')
print(f'WER tốt nhất:     {min(wers):.4f}  ({min(wers)*100:.2f}%)')
print(f'WER tệ nhất:      {max(wers):.4f}  ({max(wers)*100:.2f}%)')
print(f'')
print(f'CER trung bình:   {np.mean(cers):.4f}  ({np.mean(cers)*100:.2f}%)')
print(f'CER trung vị:     {np.median(cers):.4f}  ({np.median(cers)*100:.2f}%)')

print('\nPhân bố WER:')
for t in [0.1, 0.2, 0.3, 0.5]:
    pct = sum(1 for w in wers if w <= t) / len(wers) * 100
    print(f'  WER ≤ {int(t*100):2d}%: {pct:.1f}% mẫu')

In [ ]:
sorted_results = sorted(results, key=lambda x: x['wer'])

print('TOP 5 WER THẤP NHẤT:')
for r in sorted_results[:5]:
    print(f"  [{r['id']}] WER={r['wer']:.3f} | {r['n_chunks']} chunks")
    print(f"    Ref: {r['llm_answer'][:80]}...")
    print(f"    Hyp: {r['transcript'][:80]}...")
    print()

print('TOP 5 WER CAO NHẤT:')
for r in sorted_results[-5:][::-1]:
    print(f"  [{r['id']}] WER={r['wer']:.3f} | {r['n_chunks']} chunks")
    print(f"    Ref: {r['llm_answer'][:80]}...")
    print(f"    Hyp: {r['transcript'][:80]}...")
    print()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# WER distribution
axes[0].hist(wers, bins=30, edgecolor='black', color='steelblue')
axes[0].axvline(np.mean(wers), color='red', linestyle='--', label=f'Mean={np.mean(wers):.3f}')
axes[0].axvline(np.median(wers), color='orange', linestyle='--', label=f'Median={np.median(wers):.3f}')
axes[0].set_xlabel('WER'); axes[0].set_ylabel('Số mẫu')
axes[0].set_title('Phân phối WER'); axes[0].legend()

# CER distribution
axes[1].hist(cers, bins=30, edgecolor='black', color='darkorange')
axes[1].axvline(np.mean(cers), color='red', linestyle='--', label=f'Mean={np.mean(cers):.3f}')
axes[1].set_xlabel('CER'); axes[1].set_ylabel('Số mẫu')
axes[1].set_title('Phân phối CER'); axes[1].legend()

# WER vs n_chunks
n_chunks_list = [r['n_chunks'] for r in results]
axes[2].scatter(n_chunks_list, wers, alpha=0.5, color='green')
axes[2].set_xlabel('Số chunks'); axes[2].set_ylabel('WER')
axes[2].set_title('WER theo số chunks')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'wer_distribution.png'), dpi=150)
plt.show()

In [ ]:
# Lưu về Google Drive (tuỳ chọn)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(RESULTS_FILE, '/content/drive/MyDrive/tts_wer_results.jsonl')
# shutil.copy(os.path.join(OUTPUT_DIR, 'wer_distribution.png'), '/content/drive/MyDrive/')
# print('Đã lưu về Google Drive')